# tensor-unbind — ex9: verify gradients flow through unbind to per-slot gradients

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `tensor-unbind`. Running the final beacon cell reports progress against the `Numpy: Indexing and selection` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Numpy: Indexing and selection` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`tensor-unbind`** (exercise 9). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "tensor-unbind"
DD_SUBTOPIC = "Numpy: Indexing and selection"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## torch.unbind through autograd — quick refresher

`x.unbind(dim=k)` returns a tuple of **view** tensors that share storage with `x`. When `x.requires_grad=True`, the view tuple is fully differentiable: each element has its own `grad_fn`, and gradients accumulated through any element flow back to `x` through the slot it came from.

**Compared to indexed access.** `x[0]`, `x[1]`, ... also produce differentiable views, but `unbind` gives you the full tuple in one call — handy when you want symbolic per-slice names and want to verify the backward graph treats them as parallel branches.

**This drill (ex9) vs ex1-8.** Earlier exercises destructured tensors for forward computation (ray casting, attention heads, RGB → grayscale). ex9 takes the same destructure and runs `.backward()` through it to verify gradients flow correctly to each slot of the source tensor.

### Exercise 9 — verify gradients flow through unbind to per-slot gradients

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Analyze
> LO: Analyze the autograd graph produced by `unbind` by running `.backward()` through a per-slot weighted sum and verifying that each row of `x.grad` matches the analytic per-slot weight.
> Keywords: autograd, backward, per-slot-grad, view-graph
> ```

**KCs targeted:** `unbind-returns-views`, `unbind-autograd-flow`

Implement `ex9_unbind_grad_check(x, weights)`.

Given `x` of shape `(N, D)` with `requires_grad=True` and a 1-D `weights` tensor of length `N`, do the following:

1. Use `x.unbind(dim=0)` to get a tuple of `N` row-views `r_0, r_1, ..., r_{N-1}`, each shape `(D,)`.
2. Compute the scalar loss `L = sum_i weights[i] * r_i.sum()`. Build it by iterating over the unbound tuple — DO NOT collapse back via `stack` first; the point is to verify autograd handles the view-tuple branching.
3. Call `L.backward()`.
4. Return `x.grad` — a `(N, D)` tensor where row `i` should equal `weights[i]` (a vector of all-`weights[i]`).

**Print** `len({id(r.grad_fn) for r in rows})` so the caller can see that each unbound row got its own `grad_fn` node (not one shared node).

Inputs: `x` `(N, D)` float, requires_grad; `weights` `(N,)` float.
Output: `x.grad` `(N, D)` float.

The visualization plots `x.grad` as a heatmap so you can see the constant-per-row pattern that proves autograd routed each slot's gradient back correctly.

In [ ]:
def ex9_unbind_grad_check(x: Tensor, weights: Tensor) -> Tensor:
    rows = x.unbind(dim=0)
    # Hold all grad_fn refs simultaneously to count distinct objects
    # (without this list, Python may recycle the same id slot between calls).
    fn_refs = [r.grad_fn for r in rows]
    distinct = len({id(fn) for fn in fn_refs})
    print(f'  distinct grad_fn nodes across unbound rows: {distinct}')
    loss = sum(weights[i] * rows[i].sum() for i in range(len(rows)))
    loss.backward()
    return x.grad


<details><summary>Solution</summary>

```python
def ex9_unbind_grad_check(x: Tensor, weights: Tensor) -> Tensor:
    rows = x.unbind(dim=0)
    # Hold all grad_fn refs simultaneously to count distinct objects
    # (without this list, Python may recycle the same id slot between calls).
    fn_refs = [r.grad_fn for r in rows]
    distinct = len({id(fn) for fn in fn_refs})
    print(f'  distinct grad_fn nodes across unbound rows: {distinct}')
    loss = sum(weights[i] * rows[i].sum() for i in range(len(rows)))
    loss.backward()
    return x.grad
```

**Why each row has its own `grad_fn`.** `unbind` is implemented as a set of `select` views, each registered with the autograd engine as a separate UnbindBackward / SelectBackward node. The print shows `N` distinct ids, confirming the branching — if it ever showed `1` you'd know the framework was sharing a node and you'd need to investigate aliasing.

**Why row `i` of `x.grad` equals `weights[i]`.** `L = sum_i w_i * sum_j x_ij`. Then `∂L/∂x_ij = w_i` for every `j`. Hence each row of the gradient is a constant equal to its weight — exactly what the heatmap shows.

**Edge case worth knowing.** If you accumulate gradients twice (call backward on a fresh loss without zeroing `x.grad` first), the values will add. The drill returns the post-backward gradient directly; in training code you'd `optimizer.zero_grad()` before each step.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex9'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex9',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()